# Sesión 11 · Analiza datos a lo largo del tiempo

Badge 2 · Analista con Pandas

Ordena fechas, distingue frecuencia de agregación y calcula indicadores temporales sin mirar el futuro.

150 minutos de clase y 45–60 minutos de tarea. Ejecuta en orden.

## Fechas, orden y zona horaria

Convierte fechas con un formato explícito y revisa errores antes de crear un DatetimeIndex. Ordena cronológicamente: una ventana por tiempo depende del orden y la duración, no del orden de llegada del archivo.

tz_localize asigna una zona a fechas sin zona; tz_convert transforma instantes que ya tienen zona. Define la zona de la fuente antes de juntar datos; utc=True no debe reinterpretar sin análisis una hora local.

In [1]:
import pandas as pd
df=pd.DataFrame({"Fecha":["2026-01-03","2026-01-01"],"Ingreso":[30,10]})
df["Fecha"]=pd.to_datetime(df.Fecha,format="%Y-%m-%d")
s=df.set_index("Fecha").sort_index()["Ingreso"]
print(s)
print(s.index.tz_localize("America/Costa_Rica").tz_convert("UTC"))

Fecha
2026-01-01    10
2026-01-03    30
Name: Ingreso, dtype: int64
DatetimeIndex(['2026-01-01 06:00:00+00:00', '2026-01-03 06:00:00+00:00'], dtype='datetime64[us, UTC]', name='Fecha', freq=None)


**Qué observar:** Primero aparece enero 1; medianoche de Costa Rica corresponde a 06:00 UTC en estas fechas.

**Recuerda:** Conserva la definición de fecha local o instante: no representan lo mismo.

## resample agrega; asfreq cambia la grilla

resample agrupa observaciones en intervalos temporales y requiere una agregación. asfreq coloca la serie sobre una frecuencia y puede crear huecos sin resumir valores.

Define qué significa un día sin registro. sum(min_count=1) conserva como faltante un intervalo vacío. Para promedios diarios, los días desconocidos no deben convertirse en días con cero ventas sin justificación.

In [2]:
import pandas as pd
s=pd.Series([10,30],index=pd.to_datetime(["2026-01-01","2026-01-03"]))
print(s.asfreq("D"))
print(s.resample("D").sum(min_count=1))
print(s.resample("MS").sum(min_count=1))

2026-01-01    10.0
2026-01-02     NaN
2026-01-03    30.0
Freq: D, dtype: float64
2026-01-01    10.0
2026-01-02     NaN
2026-01-03    30.0
Freq: D, dtype: float64
2026-01-01    40
Freq: MS, dtype: int64


**Qué observar:** Enero 2 aparece como NaN; el ingreso observado de enero suma 40.

**Recuerda:** Informa cobertura y períodos parciales; un total mensual no garantiza que todos los días estén observados.

## Rezagos, ventanas y acumulados

shift(1) desplaza valores una observación y permite crear un rezago. rolling(3) usa tres observaciones; rolling("3D") usa una duración temporal y puede incluir cantidades distintas de registros. min_periods controla cuántos datos válidos se requieren.

expanding usa todos los datos disponibles hasta cada punto. Para una variable que pretende predecir el presente, calcula estadísticas con datos anteriores: shift(1).rolling(...). Evita ventanas centradas que incorporen el futuro.

In [3]:
import pandas as pd
s=pd.Series([10,20,30,40],index=pd.date_range("2026-01-01",periods=4,freq="D"))
print(s.rolling(3,min_periods=3).mean())
print(s.shift(1).rolling(2,min_periods=2).mean())
print(s.expanding().mean())

2026-01-01     NaN
2026-01-02     NaN
2026-01-03    20.0
2026-01-04    30.0
Freq: D, dtype: float64
2026-01-01     NaN
2026-01-02     NaN
2026-01-03    15.0
2026-01-04    25.0
Freq: D, dtype: float64
2026-01-01    10.0
2026-01-02    15.0
2026-01-03    20.0
2026-01-04    25.0
Freq: D, dtype: float64


**Qué observar:** Media de 3: NaN, NaN, 20, 30. Media de 2 anteriores: NaN, NaN, 15, 25. Acumulada: 10, 15, 20, 25.

**Recuerda:** Ventanas por observaciones y por tiempo solo coinciden bajo ciertas frecuencias y ausencia de huecos.

## Reto · Encuentra el día faltante

Reindexa a días la serie con enero 1=10 y enero 3=30.

**Criterio:** Enero 2 queda como NaN.

In [ ]:
# Tu solución aquí.

## Reto · Promedio sin futuro

Calcula la media de dos observaciones anteriores para [10,20,30,40].

**Criterio:** NaN, NaN, 15, 25.

In [ ]:
# Tu solución aquí.

## Tarea · Laboratorio · Series de tiempo

Ordena fechas, distingue frecuencia de agregación y calcula indicadores temporales sin mirar el futuro.

1. Genera 10 días con ingresos y elimina un día deliberadamente.
2. Compara asfreq diario y resample semanal; declara cómo tratas huecos.
3. Calcula media móvil de tres observaciones, media de tres días y acumulada.
4. Grafica y explica diferencias, cobertura y riesgo de fuga de información.

In [ ]:
# Desarrolla tu tarea y explica los resultados.